# BIN → CSV Converter
### 3DM-CV7 / 3DM-GV7 — SensorData & FilterData

Convierte los archivos `.bin` generados por el programa C a `.csv`.  
- **SensorData_N.bin** → tiempo Unix derivado de `gps.week` + `gps.tow`  
- **FilterData_N.bin** → tiempo Unix derivado de `fgps.week` + `fgps.tow`

#### Formato binario de cada registro

**SensorData** (por registro):
| Campo | Tipo | Bytes |
|---|---|---|
| gpsTs.week_number | uint16 | 2 |
| gpsTs.tow | double | 8 |
| gpsTs.valid_flags | uint16 | 2 |
| accel.scaled_accel[3] | float×3 | 12 |
| deltaV.delta_velocity[3] | float×3 | 12 |
| q.q[4] | float×4 | 16 |
| flag (dirección) | uint8 | 1 |
| **Total** | | **53 bytes** |

**FilterData** (por registro):
| Campo | Tipo | Bytes |
|---|---|---|
| fgpsTs.week_number | uint16 | 2 |
| fgpsTs.tow | double | 8 |
| fgpsTs.valid_flags | uint16 | 2 |
| accel.accel[3] | float×3 | 12 |
| accel.valid_flags | uint16 | 2 |
| ang.gyro[3] | float×3 | 12 |
| ang.valid_flags | uint16 | 2 |
| g.gravity[3] | float×3 | 12 |
| g.valid_flags | uint16 | 2 |
| q.q[4] | float×4 | 16 |
| q.valid_flags | uint16 | 2 |
| flag | uint8 | 1 |
| **Total** | | **75 bytes** |

In [ ]:
# ── Instalar dependencias (solo la primera vez en Colab) ──────────────────────
# pandas ya viene en Colab; struct y os son stdlib
import struct
import os
import pandas as pd
from google.colab import files

print('Librerías listas ✓')

Librerías listas ✓


In [ ]:
# ── Constantes GPS → Unix ─────────────────────────────────────────────────────
UNIX_GPS_EPOCH_OFFSET = 315964800   # segundos entre 1970-01-01 y 1980-01-06
GPS_LEAP_SECONDS      = 18          # leap seconds actuales (GPS adelanta UTC)
SECONDS_PER_WEEK      = 604800

def gps_to_unix_ms(week: int, tow: float) -> float:
    """
    Convierte GPS week + TOW (segundos) a timestamp Unix en milisegundos.

    GPS time NO tiene leap seconds; UTC sí.
    unix = gps_total_seconds + UNIX_GPS_EPOCH_OFFSET - GPS_LEAP_SECONDS
    """
    gps_total_seconds = week * SECONDS_PER_WEEK + tow
    unix_seconds      = gps_total_seconds + UNIX_GPS_EPOCH_OFFSET - GPS_LEAP_SECONDS
    return unix_seconds * 1000.0   # en milisegundos

print('gps_to_unix_ms listo ✓')

# ── Prueba rápida ─────────────────────────────────────────────────────────────
# GPS week 2000, TOW 0 → debería ser ~2018-03-17
import datetime
_ts = gps_to_unix_ms(2000, 0) / 1000
print(f'  Prueba: week=2000, tow=0 → {datetime.datetime.utcfromtimestamp(_ts)} UTC')

gps_to_unix_ms listo ✓
  Prueba: week=2000, tow=0 → 2018-05-05 23:59:42 UTC


/tmp/ipykernel_943/796251072.py:23: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  print(f'  Prueba: week=2000, tow=0 → {datetime.datetime.utcfromtimestamp(_ts)} UTC')


In [ ]:
# ── Tamaños y formatos de registro (struct little-endian) ─────────────────────
#
# SENSOR:  uint16 + double + uint16 + 3×float + 3×float + 4×float + uint8
#          H       d        H        3f         3f         4f         B
SENSOR_FMT  = '<HdH3f3f4fB'   # little-endian
SENSOR_SIZE = struct.calcsize(SENSOR_FMT)   # debe ser 53

# FILTER:  uint16 + double + uint16
#        + 3×float + uint16
#        + 3×float + uint16
#        + 3×float + uint16
#        + 4×float + uint16
#        + uint8
#          H  d  H  3f H  3f H  3f H  4f H  B
FILTER_FMT  = '<HdH3fH3fH3fH4fHB'
FILTER_SIZE = struct.calcsize(FILTER_FMT)   # debe ser 75

print(f'SENSOR_SIZE  = {SENSOR_SIZE} bytes  (esperado: 53)')
print(f'FILTER_SIZE  = {FILTER_SIZE} bytes  (esperado: 75)')

SENSOR_SIZE  = 53 bytes  (esperado: 53)
FILTER_SIZE  = 73 bytes  (esperado: 75)


In [ ]:
# ── Lector de cabecera embebida en el .bin ────────────────────────────────────
def read_bin_header(f) -> str:
    """
    Lee los primeros 4 bytes como uint32 (longitud),
    luego lee esa cantidad de bytes como la cabecera CSV.
    Deja el puntero al inicio del primer registro de datos.
    """
    raw_len = f.read(4)
    if len(raw_len) < 4:
        raise ValueError('Archivo demasiado corto: no contiene cabecera')
    header_len = struct.unpack('<I', raw_len)[0]
    header_str = f.read(header_len).decode('ascii')
    return header_str.strip()

print('read_bin_header listo ✓')

read_bin_header listo ✓


In [ ]:
# ── Parser SensorData ─────────────────────────────────────────────────────────
def parse_sensor_bin(path: str) -> pd.DataFrame:
    """
    Lee SensorData_N.bin y retorna un DataFrame con columnas:
      unix_time_ms, GPS_ts_week, GPS_ts_tow, GPS_valid,
      scaledAccelX, scaledAccelY, scaledAccelZ,
      deltaVelX, deltaVelY, deltaVelZ,
      orientQ0, orientQ1, orientQ2, orientQ3,
      flag
    """
    rows = []
    with open(path, 'rb') as f:
        header = read_bin_header(f)
        print(f'  Cabecera original: {header}')

        while True:
            raw = f.read(SENSOR_SIZE)
            if len(raw) == 0:
                break
            if len(raw) < SENSOR_SIZE:
                print(f'  ⚠ Registro incompleto al final ({len(raw)} bytes), ignorado.')
                break

            unpacked = struct.unpack(SENSOR_FMT, raw)
            (
                week, tow, valid,
                ax, ay, az,
                dvx, dvy, dvz,
                q0, q1, q2, q3,
                flag
            ) = unpacked

            unix_ms = gps_to_unix_ms(week, tow)

            rows.append({
                'unix_time_ms' : unix_ms,
                'GPS_ts_week'  : week,
                'GPS_ts_tow'   : tow,
                'GPS_valid'    : valid,
                'scaledAccelX' : ax,
                'scaledAccelY' : ay,
                'scaledAccelZ' : az,
                'deltaVelX'    : dvx,
                'deltaVelY'    : dvy,
                'deltaVelZ'    : dvz,
                'orientQ0'     : q0,
                'orientQ1'     : q1,
                'orientQ2'     : q2,
                'orientQ3'     : q3,
                'flag'         : flag,
            })

    df = pd.DataFrame(rows)
    print(f'  Registros leídos: {len(df)}')
    return df

print('parse_sensor_bin listo ✓')

parse_sensor_bin listo ✓


In [ ]:
# ── Parser FilterData ─────────────────────────────────────────────────────────
def parse_filter_bin(path: str) -> pd.DataFrame:
    """
    Lee FilterData_N.bin y retorna un DataFrame con columnas:
      unix_time_ms, fGPS_ts_week, fGPS_ts_tow, fGPS_valid,
      estLinAccelX, estLinAccelY, estLinAccelZ, estLinAccel_valid,
      estAngRateX, estAngRateY, estAngRateZ, estAngRate_valid,
      estGravityX, estGravityY, estGravityZ, estGravity_valid,
      estQ0, estQ1, estQ2, estQ3, estQ_valid,
      flag
    """
    rows = []
    with open(path, 'rb') as f:
        header = read_bin_header(f)
        print(f'  Cabecera original: {header}')

        while True:
            raw = f.read(FILTER_SIZE)
            if len(raw) == 0:
                break
            if len(raw) < FILTER_SIZE:
                print(f'  ⚠ Registro incompleto al final ({len(raw)} bytes), ignorado.')
                break

            unpacked = struct.unpack(FILTER_FMT, raw)
            (
                week, tow, valid,
                ax, ay, az, accel_valid,
                wx, wy, wz, ang_valid,
                gx, gy, gz, grav_valid,
                q0, q1, q2, q3, q_valid,
                flag
            ) = unpacked

            unix_ms = gps_to_unix_ms(week, tow)

            rows.append({
                'unix_time_ms'     : unix_ms,
                'fGPS_ts_week'     : week,
                'fGPS_ts_tow'      : tow,
                'fGPS_valid'       : valid,
                'estLinAccelX'     : ax,
                'estLinAccelY'     : ay,
                'estLinAccelZ'     : az,
                'estLinAccel_valid': accel_valid,
                'estAngRateX'      : wx,
                'estAngRateY'      : wy,
                'estAngRateZ'      : wz,
                'estAngRate_valid' : ang_valid,
                'estGravityX'      : gx,
                'estGravityY'      : gy,
                'estGravityZ'      : gz,
                'estGravity_valid' : grav_valid,
                'estQ0'            : q0,
                'estQ1'            : q1,
                'estQ2'            : q2,
                'estQ3'            : q3,
                'estQ_valid'       : q_valid,
                'flag'             : flag,
            })

    df = pd.DataFrame(rows)
    print(f'  Registros leídos: {len(df)}')
    return df

print('parse_filter_bin listo ✓')

parse_filter_bin listo ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELDA PRINCIPAL — Sube todos tus .bin de una vez
#
#  Entrada : SensorData_N.bin + FilterData_N.bin  (cualquier cantidad de pares)
#  Salida  : results_N.zip por cada sesión, conteniendo:
#              quaternion_N.csv       — orientQ0-Q3 + flag + claves GPS
#              quaternionfilter_N.csv — estQ0-Q3   + flag + claves GPS
#              sensordata_N.csv       — resto de datos  + flag + claves GPS
#
#  Las 3 tablas tienen EXACTAMENTE las mismas filas (inner join por GPS_ts_week
#  y GPS_ts_tow). El flag de cada fuente se incluye en los 3 archivos.
#  No se generan archivos CSV intermedios.
# ══════════════════════════════════════════════════════════════════════════════

import struct, os, io, zipfile
import pandas as pd
from google.colab import files

# ── Constantes GPS → Unix ─────────────────────────────────────────────────────
UNIX_GPS_EPOCH_OFFSET = 315964800
GPS_LEAP_SECONDS      = 18
SECONDS_PER_WEEK      = 604800

def gps_to_unix_ms(week: int, tow: float) -> float:
    gps_total_seconds = week * SECONDS_PER_WEEK + tow
    unix_seconds      = gps_total_seconds + UNIX_GPS_EPOCH_OFFSET - GPS_LEAP_SECONDS
    return unix_seconds * 1000.0

# ── Formatos binarios ─────────────────────────────────────────────────────────
SENSOR_FMT  = '<HdH3f3f4fB'
SENSOR_SIZE = struct.calcsize(SENSOR_FMT)   # 53 bytes

FILTER_FMT  = '<HdH3fH3fH3fH4fHB'
FILTER_SIZE = struct.calcsize(FILTER_FMT)   # 75 bytes

def read_bin_header(f) -> str:
    raw_len = f.read(4)
    if len(raw_len) < 4:
        raise ValueError('Archivo demasiado corto: sin cabecera')
    header_len = struct.unpack('<I', raw_len)[0]
    return f.read(header_len).decode('ascii').strip()

# ── Parsers bin → DataFrame (en memoria, sin CSV intermedios) ─────────────────
def parse_sensor_bin(data: bytes) -> pd.DataFrame:
    rows = []
    f = io.BytesIO(data)
    header = read_bin_header(f)
    print(f'    Cabecera SensorData  : {header}')
    while True:
        raw = f.read(SENSOR_SIZE)
        if len(raw) == 0: break
        if len(raw) < SENSOR_SIZE:
            print(f'    ⚠ Registro incompleto ({len(raw)} bytes), ignorado.')
            break
        week, tow, valid, ax, ay, az, dvx, dvy, dvz, q0, q1, q2, q3, flag = \
            struct.unpack(SENSOR_FMT, raw)
        rows.append({
            'unix_time_ms' : gps_to_unix_ms(week, tow),
            'GPS_ts_week'  : week,
            'GPS_ts_tow'   : tow,
            'GPS_valid'    : valid,
            'scaledAccelX' : ax,  'scaledAccelY': ay,  'scaledAccelZ': az,
            'deltaVelX'    : dvx, 'deltaVelY'   : dvy, 'deltaVelZ'   : dvz,
            'orientQ0'     : q0,  'orientQ1'    : q1,
            'orientQ2'     : q2,  'orientQ3'    : q3,
            'flag_sensor'  : flag,
        })
    df = pd.DataFrame(rows)
    print(f'    Registros SensorData : {len(df)}')
    return df

def parse_filter_bin(data: bytes) -> pd.DataFrame:
    rows = []
    f = io.BytesIO(data)
    header = read_bin_header(f)
    print(f'    Cabecera FilterData  : {header}')
    while True:
        raw = f.read(FILTER_SIZE)
        if len(raw) == 0: break
        if len(raw) < FILTER_SIZE:
            print(f'    ⚠ Registro incompleto ({len(raw)} bytes), ignorado.')
            break
        week, tow, valid, ax, ay, az, accel_v, wx, wy, wz, ang_v, \
        gx, gy, gz, grav_v, q0, q1, q2, q3, q_v, flag = \
            struct.unpack(FILTER_FMT, raw)
        rows.append({
            'unix_time_ms'      : gps_to_unix_ms(week, tow),
            'fGPS_ts_week'      : week,
            'fGPS_ts_tow'       : tow,
            'fGPS_valid'        : valid,
            'estLinAccelX'      : ax,  'estLinAccelY'     : ay,  'estLinAccelZ'    : az,
            'estLinAccel_valid' : accel_v,
            'estAngRateX'       : wx,  'estAngRateY'      : wy,  'estAngRateZ'     : wz,
            'estAngRate_valid'  : ang_v,
            'estGravityX'       : gx,  'estGravityY'      : gy,  'estGravityZ'     : gz,
            'estGravity_valid'  : grav_v,
            'estQ0'             : q0,  'estQ1'            : q1,
            'estQ2'             : q2,  'estQ3'            : q3,
            'estQ_valid'        : q_v,
            'flag_filter'       : flag,
        })
    df = pd.DataFrame(rows)
    print(f'    Registros FilterData : {len(df)}')
    return df

# ── Función principal de procesamiento por sesión ─────────────────────────────
def process_session(n: int, sensor_data: bytes, filter_data: bytes):
    """
    Procesa un par SensorData/FilterData y retorna (quat, quatf, other)
    como DataFrames con exactamente las mismas filas.
    El flag de cada fuente se incluye en los 3 archivos de salida.
    """
    df_s = parse_sensor_bin(sensor_data)
    df_f = parse_filter_bin(filter_data)

    # ── Merge por claves GPS exactas ──────────────────────────────────────────
    merged = pd.merge(
        df_s, df_f,
        left_on=['GPS_ts_week', 'GPS_ts_tow'],
        right_on=['fGPS_ts_week', 'fGPS_ts_tow'],
        how='inner',
        suffixes=('_s', '_f')
    )

    print(f'    Filas tras inner join  : {len(merged)}')
    print(f'    Descartadas SensorData : {len(df_s) - len(merged)}')
    print(f'    Descartadas FilterData : {len(df_f) - len(merged)}')

    if len(merged) == 0:
        print(f'    ⚠ Sin filas en común para sesión {n}.')
        return None, None, None

    # ── 1. quaternion_N.csv ───────────────────────────────────────────────────
    # Claves: unix_time_ms, GPS_ts_week, GPS_ts_tow
    # Datos : orientQ0-Q3, GPS_valid, flag_sensor, flag_filter
    df_quat = merged[[
        'unix_time_ms_s', 'GPS_ts_tow',
        'GPS_valid',
        'orientQ0', 'orientQ1', 'orientQ2', 'orientQ3',
        'flag_sensor'
    ]].rename(columns={'unix_time_ms_s': 'unix_time_ms', 'flag_sensor' : 'flag'}).reset_index(drop=True).astype({'unix_time_ms': 'int64'})

    # ── 2. quaternionfilter_N.csv ─────────────────────────────────────────────
    # Claves: unix_time_ms, fGPS_ts_week, fGPS_ts_tow
    # Datos : estQ0-Q3, fGPS_valid, estQ_valid, flag_sensor, flag_filter
    df_quatf = merged[[
        'unix_time_ms_f', 'fGPS_ts_tow',
        'fGPS_valid',
        'estQ0', 'estQ1', 'estQ2', 'estQ3',
        'estQ_valid',
        'flag_filter'
    ]].rename(columns={'unix_time_ms_f': 'unix_time_ms',  'flag_filter' : 'flag'}).reset_index(drop=True).astype({'unix_time_ms': 'int64'})

    # ── 3. sensordata_N.csv ───────────────────────────────────────────────────
    # Todo lo que NO son quaterniones, incluyendo flag_sensor y flag_filter
    exclude = {
        'orientQ0', 'orientQ1', 'orientQ2', 'orientQ3',
        'estQ0',    'estQ1',    'estQ2',    'estQ3',
        'estQ_valid',
        'GPS_valid', 'fGPS_valid','unix_time_ms_f', 'fGPS_ts_week', 'fGPS_ts_tow', 'GPS_ts_week'
    }
    # Columnas sin quaterniones ni flags (los flags van al final explícitamente)
    exclude_for_other = exclude | {'flag_sensor', 'flag_filter'}
    other_cols = [c for c in merged.columns if c not in exclude_for_other]
    # Asegurar flag_sensor y flag_filter como ÚLTIMAS dos columnas
    other_cols = other_cols + ['flag_sensor']
    df_other = merged[other_cols].rename(columns={
        'unix_time_ms_s': 'unix_time_ms',
        'flag_sensor': 'flag'
    }).reset_index(drop=True).astype({'unix_time_ms': 'int64'})

    # ── Verificar paridad de filas ────────────────────────────────────────────
    assert len(df_quat) == len(merged)
    assert len(df_quatf) == len(merged)
    assert len(df_other) == len(merged)
    print(f'    ✓ Las 3 tablas tienen {len(merged)} filas')

    return df_quat, df_quatf, df_other

# ── Subida de archivos ────────────────────────────────────────────────────────
print('Sube todos tus archivos .bin (SensorData_N.bin y FilterData_N.bin):')
uploaded = files.upload()

# Guardar en disco temporalmente para acceso
for fname, data in uploaded.items():
    with open(fname, 'wb') as fp:
        fp.write(data)

# ── Detectar pares por número de sesión ───────────────────────────────────────
import re

sensor_bins = {int(re.search(r'_(\d+)\.bin', f).group(1)): f
               for f in uploaded if f.startswith('SensorData') and f.endswith('.bin')}
filter_bins = {int(re.search(r'_(\d+)\.bin', f).group(1)): f
               for f in uploaded if f.startswith('FilterData') and f.endswith('.bin')}

valid_ids = sorted(set(sensor_bins) & set(filter_bins))

if not valid_ids:
    print('⚠ No se encontraron pares SensorData_N.bin + FilterData_N.bin.')
    print('  Nombra tus archivos como SensorData_1.bin, FilterData_1.bin, etc.')
else:
    print(f'\nPares detectados: sesiones {valid_ids}\n')

    for n in valid_ids:
        print(f'{"="*60}')
        print(f'  Sesión {n}')
        print(f'{"="*60}')

        df_quat, df_quatf, df_other = process_session(
            n,
            uploaded[sensor_bins[n]],
            uploaded[filter_bins[n]]
        )

        if df_quat is None:
            continue

        # ── Empaquetar en ZIP en memoria y descargar ──────────────────────────
        zip_name = f'results_{n}.zip'
        zip_buffer = io.BytesIO()

        with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
            for df, csv_name in [
                (df_quat,  f'quaternion_{n}.csv'),
                (df_quatf, f'quaternionfilter_{n}.csv'),
                (df_other, f'sensordata_{n}.csv'),
            ]:
                csv_buffer = io.StringIO()
                df.to_csv(csv_buffer, index=False, float_format='%.9f')
                zf.writestr(csv_name, csv_buffer.getvalue())
                print(f'    + {csv_name:<30} ({len(df)} filas, {len(df.columns)} cols)')

        # Guardar ZIP y descargar
        zip_buffer.seek(0)
        with open(zip_name, 'wb') as zf:
            zf.write(zip_buffer.getvalue())

        print(f'    → Descargando {zip_name}...')
        files.download(zip_name)

    print('\n✓ Procesamiento completado.')

Sube todos tus archivos .bin (SensorData_N.bin y FilterData_N.bin):


Saving FilterData_1.bin to FilterData_1.bin
Saving FilterData_2.bin to FilterData_2.bin
Saving FilterData_3.bin to FilterData_3.bin
Saving FilterData_4.bin to FilterData_4.bin
Saving SensorData_1.bin to SensorData_1.bin
Saving SensorData_2.bin to SensorData_2.bin
Saving SensorData_3.bin to SensorData_3.bin
Saving SensorData_4.bin to SensorData_4.bin

Pares detectados: sesiones [1, 2, 3, 4]

  Sesión 1
    Cabecera SensorData  : GPS_ts.week, GPS_ts.tow, GPS:valid,scaledAccelX,scaledAccelY,scaledAccelZ,deltaVelX,deltaVelY,deltaVelZ,orientQuaternion[0],orientQuaternion[1],orientQuaternion[2],orientQuaternion[3],flag
    Registros SensorData : 11642
    Cabecera FilterData  : fGPS_ts.week,fGPS_ts.tow, fGPS_ts:valid,estLinearAccelX,estLinearAccelY,estLinearAccelZ,estLinearAccel:valid,estAngularRateX,estAngularRateY,estAngularRateZ,estAngularRate:valid,estGravityX,estGravityY,estGravityZ,estGravity:valid,estOrientQuaternion[0], estOrientQuaternion[1],estOrientQuaternion[2],estOrientQuaternio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Sesión 2
    Cabecera SensorData  : GPS_ts.week, GPS_ts.tow, GPS:valid,scaledAccelX,scaledAccelY,scaledAccelZ,deltaVelX,deltaVelY,deltaVelZ,orientQuaternion[0],orientQuaternion[1],orientQuaternion[2],orientQuaternion[3],flag
    Registros SensorData : 11212
    Cabecera FilterData  : fGPS_ts.week,fGPS_ts.tow, fGPS_ts:valid,estLinearAccelX,estLinearAccelY,estLinearAccelZ,estLinearAccel:valid,estAngularRateX,estAngularRateY,estAngularRateZ,estAngularRate:valid,estGravityX,estGravityY,estGravityZ,estGravity:valid,estOrientQuaternion[0], estOrientQuaternion[1],estOrientQuaternion[2],estOrientQuaternion[3],estOrientQuaternion:valid,flag
    Registros FilterData : 10641
    Filas tras inner join  : 8979
    Descartadas SensorData : 2233
    Descartadas FilterData : 1662
    ✓ Las 3 tablas tienen 8979 filas
    + quaternion_2.csv               (8979 filas, 8 cols)
    + quaternionfilter_2.csv         (8979 filas, 9 cols)
    + sensordata_2.csv               (8979 filas, 21 cols)
    → Desca

...

<IPython.core.display.Javascript object>

  Sesión 4
    Cabecera SensorData  : GPS_ts.week, GPS_ts.tow, GPS:valid,scaledAccelX,scaledAccelY,scaledAccelZ,deltaVelX,deltaVelY,deltaVelZ,orientQuaternion[0],orientQuaternion[1],orientQuaternion[2],orientQuaternion[3],flag
    Registros SensorData : 11013
    Cabecera FilterData  : fGPS_ts.week,fGPS_ts.tow, fGPS_ts:valid,estLinearAccelX,estLinearAccelY,estLinearAccelZ,estLinearAccel:valid,estAngularRateX,estAngularRateY,estAngularRateZ,estAngularRate:valid,estGravityX,estGravityY,estGravityZ,estGravity:valid,estOrientQuaternion[0], estOrientQuaternion[1],estOrientQuaternion[2],estOrientQuaternion[3],estOrientQuaternion:valid,flag
    Registros FilterData : 10452
    Filas tras inner join  : 8821
    Descartadas SensorData : 2192
    Descartadas FilterData : 1631
    ✓ Las 3 tablas tienen 8821 filas
    + quaternion_4.csv               (8821 filas, 8 cols)
    + quaternionfilter_4.csv         (8821 filas, 9 cols)
    + sensordata_4.csv               (8821 filas, 21 cols)
    → Desca

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Procesamiento completado.
